In [4]:
import glob
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Automatically find the exact CSV path inside Kaggle's input folder
csv_path = glob.glob("/kaggle/input/**/DataCoSupplyChainDataset.csv", recursive=True)[0]
print("Found file at:", csv_path)

df = pd.read_csv(csv_path, encoding="latin1")
print("Dataset shape:", df.shape)
print("\nDelivery Status Breakdown (%):")
print(df["Delivery Status"].value_counts(normalize=True) * 100)


Found file at: /kaggle/input/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis/DataCoSupplyChainDataset.csv
Dataset shape: (180519, 53)

Delivery Status Breakdown (%):
Delivery Status
Late delivery        54.829132
Advance shipping     23.040234
Shipping on time     17.835242
Shipping canceled     4.295393
Name: proportion, dtype: float64


In [6]:
# CELL 2: Clean & Prepare Governance/Process Features
cols = [
    "Delivery Status",
    "Days for shipment (scheduled)",
    "Shipping Mode",
    "Customer Segment",
    "Order Region",
    "Sales"
]

df_clean = df[cols].dropna().copy()
df_clean.columns = ["delivery_status", "scheduled_days", "shipping_mode", "segment", "region", "sales"]

print("Cleaned dataset shape:", df_clean.shape)
print("Missing values remaining:\n", df_clean.isna().sum())


Cleaned dataset shape: (180519, 6)
Missing values remaining:
 delivery_status    0
scheduled_days     0
shipping_mode      0
segment            0
region             0
sales              0
dtype: int64


In [11]:
# CELL 3: Statistical Inference — Multinomial Logistic Regression
# Reference category: 'Late delivery' (0)
status_map = {
    "Late delivery": 0,      # Reference category (0)
    "Advance shipping": 1,   # Comparison 1
    "Shipping on time": 2,   # Comparison 2
    "Shipping canceled": 3   # Comparison 3
}
df_clean["status_code"] = df_clean["delivery_status"].map(status_map)

# Use segment, region, and sales (avoiding structural separation in shipping_mode)
mnlogit_model = smf.mnlogit(
    "status_code ~ C(segment) + C(region) + sales",
    data=df_clean
).fit()

print(mnlogit_model.summary())
print("\nOdds Ratios / Exp(B) [1=Advance, 2=On Time, 3=Canceled vs. 0=Late Delivery]:")
print(np.exp(mnlogit_model.params).round(4))


Optimization terminated successfully.
         Current function value: 1.109652
         Iterations 7
                          MNLogit Regression Results                          
Dep. Variable:            status_code   No. Observations:               180519
Model:                        MNLogit   Df Residuals:                   180441
Method:                           MLE   Df Model:                           75
Date:                Tue, 24 Nov 2026   Pseudo R-squ.:               0.0006654
Time:                        14:40:58   Log-Likelihood:            -2.0031e+05
converged:                       True   LL-Null:                   -2.0045e+05
Covariance Type:            nonrobust   LLR p-value:                 2.670e-23
               status_code=1       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -0.5287      0.076     -6.947      0.000

In [13]:
# CELL 4: Predictive Machine Learning — Random Forest Classifier
X = pd.get_dummies(df_clean.drop(["delivery_status", "status_code"], axis=1), drop_first=True)
y = df_clean["delivery_status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)

print("Random Forest Multi-Class Classification Report:\n")
print(classification_report(y_test, preds))


Random Forest Multi-Class Classification Report:

                   precision    recall  f1-score   support

 Advance shipping       0.39      0.51      0.44      8282
    Late delivery       0.63      0.75      0.69     19797
Shipping canceled       0.06      0.01      0.01      1558
 Shipping on time       0.39      0.11      0.17      6467

         accuracy                           0.55     36104
        macro avg       0.37      0.34      0.33     36104
     weighted avg       0.51      0.55      0.51     36104

